In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df_train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
df_test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv')
df_sample_subm = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/sample_submission.csv')

In [ ]:
df_train = df_train[['id','date', 'store_nbr', 'family', 'sales']]
df_train['source'] = 'train'

df_test = df_test[['id','date', 'store_nbr', 'family']]
df_test['source'] = 'test'

In [ ]:
df_concat = pd.concat([df_train, df_test], ignore_index=True)

In [ ]:
df_concat['date'] = pd.to_datetime(df_concat['date'])
df_concat = df_concat.sort_values(['store_nbr', 'family', 'date'])

In [ ]:
# remove outliers
def cap_outliers(x):
    low = 0
    high = x.quantile(0.99)
    return x.clip(lower=low, upper=high)

df_outliers_cap = df_concat.copy()
df_outliers_cap['sales'] = df_concat.groupby(['store_nbr', 'family'])['sales'].transform(cap_outliers)

In [ ]:
def forecast_holt_winters(x):
    # Fit only if there are enough data points
    if x['sales'].notna().sum() > 30:
        try:
            model = ExponentialSmoothing(
                x['sales'].dropna(),
                trend='add',
                seasonal='add',
                seasonal_periods=7  # seasonality (7 days)
            ).fit(optimized=True)
            
            # Use fitted values and forward-fill missing ones
            fitted = model.fittedvalues
            result = x['sales'].copy()
            result.loc[result.notna()] = fitted.values  
            # fill remaining nulls with forecast
            forecast_len = result.isna().sum()
            if forecast_len > 0:
                forecast = model.forecast(forecast_len)
                result[result.isna()] = forecast.values
            return result
        except:
            return x['sales'].fillna(method='ffill')
    else:
        return x['sales'].fillna(method='ffill')

df_outliers_cap['sales'] = df_outliers_cap.groupby(['store_nbr', 'family'], group_keys=False).apply(forecast_holt_winters)

In [ ]:
df_test_final = df_outliers_cap[df_outliers_cap['source'] == 'test']
df_test_final = df_test_final[['id', 'sales']]
df_test_final = df_test_final.sort_values(['id'])

In [ ]:
assert (df_test_final['id'].values == df_sample_subm['id'].values).all(), "IDs do not match or are in different order"

In [ ]:
df_test_final.to_csv("submission.csv", index=False)